# ChaoSSM — Phase 9: the fork-in-the-road experiment

## Purpose

Phases 4/6/6b established that in the CURRENT architecture, chaos confers no benefit and training
gates it off. But that architecture may be testing chaos in the regime most hostile to it. Before
committing to a negative-result paper, this experiment gives chaos a genuinely fair chance by
isolating the FOUR ways the theory-to-architecture bridge may have been broken:

| # | Suspected failure | Factor that tests it |
|---|---|---|
| F1 | End-to-end training destroys chaos (reservoir theory needs it FIXED) | **frozen vs. trained** recurrence |
| F2 | We targeted deep chaos (λ≈+0.15); the benefit is at the EDGE (λ→0⁻) | **edge (λ→0⁻) vs. deep (λ>0)** regime |
| F3 | Memory tasks punish sensitive dependence — chaos's core property | **copy (memory) vs. forecasting (chaos-friendly)** task |
| F4 | The gate's (1−g)h path lets the model BYPASS the chaotic map | **gate vs. gate-removed** cell |

## Design

A factorial over {frozen, trained} × {edge, deep} × {gate, no-gate} × {copy, forecast}, plus a
GRU reference and a linear-recurrence control per task. Frozen arms train only the input
projection + readout (the reservoir-computing setup); the recurrent map is initialized into the
target regime and held fixed. Regime is set by the r-cap and a one-time calibration of the map's
Lyapunov exponent (edge: tune r so λ≈0⁻; deep: r=4.0).

## Decision rule (pre-registered)

- **If chaos helps in ANY cell** (a chaotic arm beats its own matched linear/periodic control with
  a bootstrap CI excluding 0) — most plausibly frozen + edge + forecast — the hypothesis is ALIVE:
  the earlier null was a regime/training artifact, not a property of chaos. → pursue the POSITIVE
  paper (Direction B); expand that cell to full power.
- **If chaos fails in EVERY cell**, including a frozen edge-of-chaos reservoir on its own
  chaos-friendly task with no gate to hide behind, the negative result is GENERAL and bulletproof
  — every "you built it wrong" escape route is closed. → the negative paper (Direction A) is now
  unassailable and worth writing.

## Scope / runtime

`FAST_CORE=True` runs the decisive subset first (frozen×{edge,deep}×{copy,forecast}, no-gate,
5 seeds) — the cells most likely to reveal a live effect — in ~one session. Set False for the full
factorial. Frozen arms are cheap (no recurrent gradients). Trained arms reuse the lambda-gated
path. The notebook prints a runtime projection before the loop.

## Setup

In [9]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field
from itertools import combinations, product
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))
def setup_results_dir(project="ChaoSSM",phase="phase9_fork"):
    try:
        from google.colab import drive; drive.mount("/content/drive",force_remount=False)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"./{project}_results/{phase}"); print(f"[info] not colab ({type(e).__name__})")
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
gpu: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
results -> /content/drive/My Drive/ChaoSSM/results/phase9_fork


## Configuration

In [10]:
SMOKE_TEST=False
FAST_CORE=True     # True: decisive frozen subset (~1 session). False: full factorial.

@dataclass
class Config:
    state_dim:int=64; d_model:int=64; n_layers:int=1; mlp_mult:int=2
    steps:int=4000; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; gamma_init:float=1.0; lr:float=1e-3
    seeds:int=5
    eval_n:int=1500; eval_seed:int=12345
    # tasks
    copy_len:int=12; vocab:int=16
    fc_length:int=64          # forecasting sequence length
    fc_horizon:int=1          # predict next value
    fc_bins:int=32            # discretize the chaotic series into bins for CE loss
    # regimes: r-cap sets the map's reachable dynamics; edge is calibrated to lambda~0-
    # regimes are defined by TARGET Lyapunov exponent, not r-cap. gamma is calibrated per cell
    # to hit these, so the regime labels are true by construction (verified in the calibration cell).
    r_cap_map:float=4.0                 # r-cap fixed; gamma is the calibration knob
    lambda_target:dict=field(default_factory=lambda: dict(edge=-0.02, deep=0.30))
    lambda_tol:float=0.05
    calib_steps:int=200; calib_warm:int=60
    lyap_window:int=4; lyap_subsample:int=3
    # factorial axes
    trainings:tuple=("frozen","trained")
    regimes:tuple=("edge","deep")
    gates:tuple=("gate","nogate")
    tasks:tuple=("copy","forecast")

CFG=Config()
if SMOKE_TEST:
    CFG.steps=40; CFG.seeds=2; CFG.eval_n=40
    print("*** SMOKE TEST ***")
# build the condition list
def build_conditions(cfg,fast):
    conds=[]
    if fast:
        # decisive subset: frozen only, both regimes, no-gate, both tasks
        for reg in cfg.regimes:
            for task in cfg.tasks:
                conds.append(dict(arm="chaos",training="frozen",regime=reg,gate="nogate",task=task))
        # matched frozen-linear control + GRU per task
        for task in cfg.tasks:
            conds.append(dict(arm="linear",training="frozen",regime="edge",gate="nogate",task=task))
            conds.append(dict(arm="gru",training="trained",regime="edge",gate="gate",task=task))
    else:
        for tr,reg,ga,task in product(cfg.trainings,cfg.regimes,cfg.gates,cfg.tasks):
            conds.append(dict(arm="chaos",training=tr,regime=reg,gate=ga,task=task))
        for tr,task in product(cfg.trainings,cfg.tasks):
            conds.append(dict(arm="linear",training=tr,regime="edge",gate="nogate",task=task))
        for task in cfg.tasks:
            conds.append(dict(arm="gru",training="trained",regime="edge",gate="gate",task=task))
    return conds
CONDS=build_conditions(CFG,FAST_CORE)
print(f"FAST_CORE={FAST_CORE}  conditions={len(CONDS)}  seeds={CFG.seeds}  total runs={len(CONDS)*CFG.seeds}")
print(json.dumps({k:v for k,v in asdict(CFG).items()},indent=2,default=str))

FAST_CORE=True  conditions=8  seeds=5  total runs=40
{
  "state_dim": 64,
  "d_model": 64,
  "n_layers": 1,
  "mlp_mult": 2,
  "steps": 4000,
  "batch": 64,
  "warmup_frac": 0.05,
  "lr_min_frac": 0.1,
  "grad_clip": 1.0,
  "gamma_init": 1.0,
  "lr": 0.001,
  "seeds": 5,
  "eval_n": 1500,
  "eval_seed": 12345,
  "copy_len": 12,
  "vocab": 16,
  "fc_length": 64,
  "fc_horizon": 1,
  "fc_bins": 32,
  "r_cap_map": 4.0,
  "lambda_target": {
    "edge": -0.02,
    "deep": 0.3
  },
  "lambda_tol": 0.05,
  "calib_steps": 200,
  "calib_warm": 60,
  "lyap_window": 4,
  "lyap_subsample": 3,
  "trainings": [
    "frozen",
    "trained"
  ],
  "regimes": [
    "edge",
    "deep"
  ],
  "gates": [
    "gate",
    "nogate"
  ],
  "tasks": [
    "copy",
    "forecast"
  ]
}


## 1. Cells -- gate-optional, freeze-able chaotic map

The chaotic map here has NO gate escape-hatch unless `gate=True`. With `gate="nogate"`, the
recurrence is purely `h <- fold(gamma*(W_mix h + r h(1-h)) + Bx)` -- chaos must load-bear. With
`gate="gate"`, the original `(1-g)h` bypass is restored (to reproduce the old behaviour as a
control). `freeze()` disables gradients on the recurrent map (reservoir-computing setup): only the
input projection B and the readout train.

In [11]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

class ChaosCell(nn.Module):
    def __init__(self,cfg,regime,use_gate):
        super().__init__(); self.N=cfg.state_dim; self.use_gate=use_gate; self.regime=regime
        self.r_cap=float(cfg.r_cap_map)
        self.w_r=nn.Parameter(torch.zeros(self.N))          # r = r_cap*sigmoid(w_r) -> r_cap/2 by default
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(cfg.gamma_init)))))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
        if use_gate:
            self.W_g=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
            self.b_g=nn.Parameter(torch.zeros(self.N))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad():
            self.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))
    def phi(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
    def step(self,h,x):
        drive=x@self.B.T
        if self.use_gate:
            g=torch.sigmoid(x@self.W_g.T+self.b_g)
            raw=g*self.phi(h)+(1-g)*h+drive
        else:
            raw=self.phi(h)+drive
        return triangle_fold(raw)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        # reservoir setup: freeze the map (w_r, W_mix, gamma[, gate]); keep B trainable as input proj
        for p_ in [self.w_r,self.W_mix,self.w_gamma]:
            p_.requires_grad_(False)
        if self.use_gate:
            self.W_g.requires_grad_(False); self.b_g.requires_grad_(False)

class LinearCell(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self):
        self.W_lin.requires_grad_(False)

class GRUCellArm(nn.Module):
    def __init__(self,cfg,**kw): super().__init__(); self.N=cfg.state_dim; self.cell=nn.GRUCell(cfg.d_model,cfg.state_dim)
    def step(self,h,x): return self.cell(x,h)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): pass

def make_cell(cfg,cond):
    if cond["arm"]=="gru": return GRUCellArm(cfg)
    if cond["arm"]=="linear": return LinearCell(cfg)
    return ChaosCell(cfg,cond["regime"],cond["gate"]=="gate")

## 2. Model (single recurrent layer + readout; embedding for copy, linear in for forecast)

In [12]:
class SeqModel(nn.Module):
    def __init__(self,cfg,cond,in_kind,vocab_or_dim,out_dim):
        super().__init__(); self.cfg=cfg; self.cond=cond; self.in_kind=in_kind
        if in_kind=="tokens": self.emb=nn.Embedding(vocab_or_dim,cfg.d_model)
        else: self.inp=nn.Linear(1,cfg.d_model)
        self.cell=make_cell(cfg,cond)
        self.ln=nn.LayerNorm(cfg.state_dim); self.readout=nn.Linear(cfg.state_dim,out_dim)
        self.achieved_lambda=float("nan")
        if cond["training"]=="frozen": self.cell.freeze_recurrence()
    def calibrate(self,cfg):
        # called AFTER .to(device) so all tensors are co-located. Calibrates gamma so the regime
        # label is TRUE (lambda hits target). Must run before freezing matters (gamma is set in place).
        if self.cond["arm"]=="chaos":
            _,self.achieved_lambda=calibrate_gamma(self.cell,cfg,self.cond["regime"],
                                                   seed=hash((self.cond["regime"],))%1000)
        return self
    def encode(self,x):
        return self.emb(x) if self.in_kind=="tokens" else self.inp(x.unsqueeze(-1))
    def forward(self,x):
        e=self.encode(x); B,T=e.shape[0],e.shape[1]
        h=self.cell.init_state(B,e.device,e.dtype); outs=[]
        for t in range(T):
            h=self.cell.step(h,e[:,t]); outs.append(self.readout(self.ln(h)))
        return torch.stack(outs,1)

## 3. Tasks

**copy** (memory task, tokens): reproduce a 12-token sequence -- rewards faithful retention,
theoretically HOSTILE to chaos.

**forecast** (chaos-friendly, continuous): next-step prediction of a logistic chaotic series
x_{t+1}=r x_t(1-x_t), r=3.9, discretized into `fc_bins` bins for a classification loss. Sensitive
dependence on the input history is an ASSET here -- if chaos helps anywhere, it should help here.

In [13]:
V=CFG.vocab; SEP,PAD=V,V+1; VOCAB_SIZE=V+2
def make_copy(cfg,batch,device,gen):
    L=cfg.copy_len; src=torch.randint(0,V,(batch,L),device=device,generator=gen)
    inp=torch.cat([src,torch.full((batch,1),SEP,device=device),torch.full((batch,L),PAD,device=device)],1)
    tgt=torch.cat([torch.full((batch,L+1),-100,device=device),src],1)
    return inp,tgt,"tokens",VOCAB_SIZE,VOCAB_SIZE

def make_forecast(cfg,batch,device,gen):
    L=cfg.fc_length; r=3.9
    x0=0.1+0.8*torch.rand(batch,1,device=device,generator=gen)
    xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X=torch.cat(xs,1)                       # (batch, L+1) in [0,1]
    inp=X[:,:L]                             # continuous input
    nxt=X[:,1:L+1]                          # next value
    tgt=torch.clamp((nxt*cfg.fc_bins).long(),0,cfg.fc_bins-1)  # binned target
    # predict at every step; no ignore
    return inp.float(),tgt,"scalar",1,cfg.fc_bins

TASK_FN=dict(copy=make_copy,forecast=make_forecast)
CHANCE=dict(copy=1.0/V, forecast=1.0/CFG.fc_bins)

def build_eval(cfg,task):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    return TASK_FN[task](cfg,cfg.eval_n,DEVICE,g)

@torch.no_grad()
def eval_acc(model,evalset,chunk=256):
    inp,tgt,_,_,_=evalset; model.eval(); c=tot=0.0
    for i in range(0,inp.shape[0],chunk):
        lg=model(inp[i:i+chunk]); tg=tgt[i:i+chunk]
        m=tg!=-100; p=lg.argmax(-1)
        c+=((p==tg)&m).float().sum().item(); tot+=m.float().sum().item()
    model.train(); return c/max(tot,1)

## 4. Regime calibration -- tune gamma so each regime hits its TARGET Lyapunov exponent

In [14]:
from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0,steps=None,warm=None):
    # driven-map lambda_max of the (frozen) chaotic map, small random drive to avoid a trivial
    # fixed point. Measures the ACTUAL dynamics of this coupled N-dim map, not scalar-logistic
    # intuition. Device is taken from the cell so it works on CPU or CUDA.
    steps=steps or cfg.calib_steps; warm=warm or cfg.calib_warm; N=cfg.state_dim
    dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z):
            r=cell.get_r(); u=z@cell.W_mix.T+r*z*(1-z); return triangle_fold(cell.get_gamma()*u)
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev)
        h=torch.clamp(h,0.0,1.0)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_gamma(cell,cfg,regime,seed=0):
    # bisection on gamma to hit lambda_target[regime]. gamma scales the fold argument, which scales
    # the map's local expansion, so lambda is monotone increasing in gamma -> bisection works.
    target=cfg.lambda_target[regime]
    lo,hi=0.05,8.0; best=None; best_err=1e9
    for _ in range(18):
        mid=math.sqrt(lo*hi)
        cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1]

# verify: build one cell per regime, calibrate, confirm lambda lands on target
print("Regime calibration (gamma tuned so lambda hits target):")
for reg in CFG.regimes:
    c=ChaosCell(CFG,reg,False).to(DEVICE)
    gam,lam=calibrate_gamma(c,CFG,reg,seed=0)
    print(f"  regime={reg:5s} target lambda={CFG.lambda_target[reg]:+.2f}  ->  gamma={gam:.3f}  measured lambda={lam:+.3f}")

Regime calibration (gamma tuned so lambda hits target):
  regime=edge  target lambda=-0.02  ->  gamma=0.499  measured lambda=-0.041
  regime=deep  target lambda=+0.30  ->  gamma=1.513  measured lambda=+0.250


## 5. Trainer (frozen arms train only input proj + readout)

In [15]:
def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

def run_condition(cond,seed,cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    # infer io from a sample batch
    inp,tgt,in_kind,in_dim,out_dim=TASK_FN[cond["task"]](cfg,4,DEVICE,gen)
    model=SeqModel(cfg,cond,in_kind,in_dim,out_dim).to(DEVICE)
    model.calibrate(cfg)   # calibrate gamma AFTER moving to device (all tensors co-located)
    evalset=build_eval(cfg,cond["task"])
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params,lr=cfg.lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(cfg,cfg.steps))
    div=False
    for step in range(cfg.steps):
        inp,tgt,_,_,_=TASK_FN[cond["task"]](cfg,cfg.batch,DEVICE,gen)
        lg=model(inp)
        L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1),ignore_index=-100)
        opt.zero_grad(set_to_none=True); L.backward()
        gn=torch.nn.utils.clip_grad_norm_(params,cfg.grad_clip).item(); opt.step(); sched.step()
        if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
    acc=eval_acc(model,evalset) if not div else float("nan")
    n_trainable=sum(p.numel() for p in params)
    return dict(**cond,seed=seed,diverged=div,acc=acc,n_trainable=n_trainable,
                achieved_lambda=round(float(model.achieved_lambda),3))

## 6. Run the factorial

In [16]:
def project(conds,cfg):
    # frozen arms ~ cheaper; assume 3 min frozen, 5 min trained per run at 4000 steps
    est=sum((3.0 if c["training"]=="frozen" else 5.0) for c in conds)*cfg.seeds*(cfg.steps/4000.0)
    print(f"runs: {len(conds)*cfg.seeds}  |  prior estimate (T4): ~{est/60:.1f} h [{est:.0f} min]")
project(CONDS,CFG)
rows=[]; t0=time.time(); tot=len(CONDS)*CFG.seeds; done=0; first=None
for ci,cond in enumerate(CONDS):
    ct0=time.time()
    for sd in range(CFG.seeds):
        r=run_condition(cond,sd,CFG); rows.append(r); done+=1
        tag=f"{cond['arm']}/{cond['training']}/{cond['regime']}/{cond['gate']}/{cond['task']}"
        print(f"[{done}/{tot}] {tag:44s} s={sd} acc={r['acc']:.4f}{' DIV' if r['diverged'] else ''}")
    if first is None:
        first=time.time()-ct0; per=first/CFG.seeds
        print(f"    [timing] first condition {first/60:.1f} min (~{per:.0f}s/run); projected ~{per*tot/60:.1f} min")
RES=pd.DataFrame(rows); savetable(RES,"phase9_fork_summary.csv")
print(f"\nwall clock: {(time.time()-t0)/60:.1f} min")

runs: 40  |  prior estimate (T4): ~2.3 h [140 min]
[1/40] chaos/frozen/edge/nogate/copy                s=0 acc=0.1363
[2/40] chaos/frozen/edge/nogate/copy                s=1 acc=0.0701
[3/40] chaos/frozen/edge/nogate/copy                s=2 acc=0.0711
[4/40] chaos/frozen/edge/nogate/copy                s=3 acc=0.0689
[5/40] chaos/frozen/edge/nogate/copy                s=4 acc=0.0807
    [timing] first condition 9.9 min (~118s/run); projected ~78.9 min
[6/40] chaos/frozen/edge/nogate/forecast            s=0 acc=0.8215
[7/40] chaos/frozen/edge/nogate/forecast            s=1 acc=0.8121
[8/40] chaos/frozen/edge/nogate/forecast            s=2 acc=0.8163
[9/40] chaos/frozen/edge/nogate/forecast            s=3 acc=0.8478
[10/40] chaos/frozen/edge/nogate/forecast            s=4 acc=0.8263
[11/40] chaos/frozen/deep/nogate/copy                s=0 acc=0.0777
[12/40] chaos/frozen/deep/nogate/copy                s=1 acc=0.0646
[13/40] chaos/frozen/deep/nogate/copy                s=2 acc=0.0786
[14/

## 7. Analysis -- does chaos beat its matched control in ANY cell?

In [17]:
def cliffs_delta(x,y):
    x=np.asarray(x); y=np.asarray(y)
    gt=sum((xi>y).sum() for xi in x); lt=sum((xi<y).sum() for xi in x)
    return (gt-lt)/(len(x)*len(y)) if len(x) and len(y) else float("nan")
def boot_ci(x,y,n=10000,seed=0):
    rng=np.random.default_rng(seed); x=np.asarray(x); y=np.asarray(y)
    if not len(x) or not len(y): return (float("nan"),float("nan"))
    d=np.array([rng.choice(x,len(x),True).mean()-rng.choice(y,len(y),True).mean() for _ in range(n)])
    return float(np.percentile(d,2.5)),float(np.percentile(d,97.5))

def accs(**sel):
    m=RES.copy()
    for k,v in sel.items(): m=m[m[k]==v]
    a=m.acc.values; return a[np.isfinite(a)]

# For each (task), compare each chaos cell against the matched frozen-linear control on that task.
cmp_rows=[]
for task in CFG.tasks:
    lin=accs(arm="linear",task=task)
    chaos_cells=RES[(RES.arm=="chaos")&(RES.task==task)][["training","regime","gate"]].drop_duplicates()
    for _,cc in chaos_cells.iterrows():
        ch=accs(arm="chaos",task=task,training=cc.training,regime=cc.regime,gate=cc.gate)
        if not len(ch) or not len(lin): continue
        md_=float(ch.mean()-lin.mean()); lo,hi=boot_ci(ch,lin); dl=cliffs_delta(ch,lin)
        cmp_rows.append(dict(task=task,cell=f"{cc.training}/{cc.regime}/{cc.gate}",
                             chaos_mean=round(float(ch.mean()),4),linear_mean=round(float(lin.mean()),4),
                             mean_diff=round(md_,4),ci_low=round(lo,4),ci_high=round(hi,4),
                             cliffs_delta=round(dl,3),chaos_helps=bool(lo>0)))
CMP=pd.DataFrame(cmp_rows); print(CMP.to_string(index=False)); savetable(CMP,"phase9_fork_contrasts.csv")

# also print GRU reference per task
for task in CFG.tasks:
    g=accs(arm="gru",task=task)
    print(f"[ref] GRU {task}: {g.mean():.4f}" if len(g) else f"[ref] GRU {task}: n/a")

    task               cell  chaos_mean  linear_mean  mean_diff  ci_low  ci_high  cliffs_delta  chaos_helps
    copy frozen/edge/nogate      0.0854       0.0621     0.0233  0.0077   0.0497           1.0         True
    copy frozen/deep/nogate      0.0728       0.0621     0.0107  0.0060   0.0150           1.0         True
forecast frozen/edge/nogate      0.8248       0.8755    -0.0507 -0.0674  -0.0326          -1.0        False
saved /content/drive/My Drive/ChaoSSM/results/phase9_fork/phase9_fork_contrasts.csv
[ref] GRU copy: 0.4934
[ref] GRU forecast: 0.9435


## 8. Figure

In [ ]:
fig,axes=plt.subplots(1,len(CFG.tasks),figsize=(6.2*len(CFG.tasks),4.6))
if len(CFG.tasks)==1: axes=[axes]
for ax,task in zip(axes,CFG.tasks):
    sub=RES[RES.task==task]
    labels=[]; means=[]; errs=[]; cols=[]
    for _,cc in sub[["arm","training","regime","gate"]].drop_duplicates().iterrows():
        a=accs(arm=cc.arm,task=task,training=cc.training,regime=cc.regime,gate=cc.gate)
        if not len(a): continue
        lab=cc.arm if cc.arm!="chaos" else f"ch·{cc.training[:2]}·{cc.regime}·{cc.gate[:2]}"
        labels.append(lab); means.append(a.mean()); errs.append(a.std())
        cols.append("tab:green" if cc.arm=="chaos" else ("tab:purple" if cc.arm=="gru" else "gray"))
    order=np.argsort(means)[::-1]
    labels=[labels[i] for i in order]; means=[means[i] for i in order]; errs=[errs[i] for i in order]; cols=[cols[i] for i in order]
    ax.bar(range(len(labels)),means,yerr=errs,capsize=3,color=cols)
    ax.axhline(CHANCE[task],color="r",ls="--",lw=1,label="chance")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels,rotation=45,ha="right",fontsize=7)
    ax.set_title(task); ax.set_ylabel("held-out accuracy"); ax.grid(alpha=.3,axis="y"); ax.legend(fontsize=8)
fig.suptitle("Phase 9 fork: does chaos help under a fair test? (green=chaos)")
fig.tight_layout(); savefig(fig,"phase9_fork_bars.png"); plt.close(fig)

## 9. Verdict -- which direction is real

In [20]:
helps=CMP[CMP.chaos_helps==True] if len(CMP) else CMP
alive=len(helps)>0
if alive:
    where="; ".join(f"{r.task}:{r.cell} (+{r.mean_diff}, CI[{r.ci_low},{r.ci_high}])" for _,r in helps.iterrows())
    decision=("HYPOTHESIS ALIVE (Direction B). Chaos beats its matched linear control in at least "
              f"one cell: {where}. The earlier null was a regime/training/task artifact, not a "
              "property of chaos. Next: expand the winning cell(s) to full power (more seeds, scale, "
              "and the natural comparison), and build the POSITIVE paper around where and why chaos "
              "helps. Do NOT write the negative paper.")
else:
    decision=("NEGATIVE RESULT IS GENERAL (Direction A). Chaos failed to beat its matched linear "
              "control in EVERY cell -- including a frozen edge-of-chaos reservoir, on its own "
              "chaos-friendly forecasting task, with no gate to hide behind. Every 'you built it "
              "wrong' escape route (training destroys it / wrong regime / wrong task / gate bypass) "
              "is now closed. The negative result is general and bulletproof -- write Direction A, "
              "with this factorial as the centerpiece that rules out the alternatives.")
verdict=dict(smoke_test=SMOKE_TEST, fast_core=FAST_CORE, n_conditions=len(CONDS), seeds=CFG.seeds,
             contrasts=CMP.to_dict("records") if len(CMP) else [],
             chaos_helps_cells=helps.to_dict("records") if len(helps) else [],
             hypothesis_alive=alive, decision=decision,
             any_diverged=bool(RES.diverged.any()))
savejson(verdict,"phase9_fork_verdict.json")
print(decision); print(); print(json.dumps(verdict,indent=2,default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase9_fork/phase9_fork_verdict.json
HYPOTHESIS ALIVE (Direction B). Chaos beats its matched linear control in at least one cell: copy:frozen/edge/nogate (+0.0233, CI[0.0077,0.0497]); copy:frozen/deep/nogate (+0.0107, CI[0.006,0.015]). The earlier null was a regime/training/task artifact, not a property of chaos. Next: expand the winning cell(s) to full power (more seeds, scale, and the natural comparison), and build the POSITIVE paper around where and why chaos helps. Do NOT write the negative paper.

{
  "smoke_test": false,
  "fast_core": true,
  "n_conditions": 8,
  "seeds": 5,
  "contrasts": [
    {
      "task": "copy",
      "cell": "frozen/edge/nogate",
      "chaos_mean": 0.0854,
      "linear_mean": 0.0621,
      "mean_diff": 0.0233,
      "ci_low": 0.0077,
      "ci_high": 0.0497,
      "cliffs_delta": 1.0,
      "chaos_helps": true
    },
    {
      "task": "copy",
      "cell": "frozen/deep/nogate",
      "chaos_mean": 0.0728,